# Disfluency Repair — Seq2Seq Fine-tune

Fine-tune `Wikidepia/IndoT5-small` on `text -> text_normalized` pairs from `data/disfluency_repair/{train,val,test}.jsonl`.

Input has fillers (eh/anu/um/hmm), repetitions, false starts, and self-corrections ("dua, eh, tiga"). Treated as text-to-text repair, not token tagging, since some repairs are replacements, not pure deletions.

In [1]:
import json
from pathlib import Path

import evaluate
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

MODEL_NAME = "Wikidepia/IndoT5-small"
DATA_DIR = Path("../data/disfluency_repair")
OUTPUT_DIR = Path("../models/disfluency_repair_indot5_small")
MAX_INPUT_LEN = 64
MAX_TARGET_LEN = 64
SEED = 42

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
device

'mps'

## Load data

In [2]:
def load_split(name: str) -> list[dict]:
    path = DATA_DIR / f"{name}.jsonl"
    with path.open() as f:
        return [json.loads(line) for line in f]


train_rows = load_split("train")
val_rows = load_split("val")
test_rows = load_split("test")

len(train_rows), len(val_rows), len(test_rows), train_rows[0]

(1600,
 200,
 200,
 {'id': 1737,
  'input': 'eh pak, anginnya berisik tadi, gue gak denger jelas',
  'target': 'eh pak anginnya berisik tadi gue gak denger jelas',
  'intent': 'repeat_request'})

In [3]:
train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)
test_ds = Dataset.from_list(test_rows)
train_ds

Dataset({
    features: ['id', 'input', 'target', 'intent'],
    num_rows: 1600
})

## Tokenizer + preprocessing

Task prefix `"perbaiki: "` primes the model for the repair task (T5-style task prefixes).

In [4]:
TASK_PREFIX = "perbaiki: "

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess(batch: dict) -> dict:
    inputs = [TASK_PREFIX + t for t in batch["input"]]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LEN, truncation=True
    )
    labels = tokenizer(
        text_target=batch["target"], max_length=MAX_TARGET_LEN, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


remove_cols = ["id", "input", "target", "intent"]
train_tok = train_ds.map(preprocess, batched=True, remove_columns=remove_cols)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=remove_cols)
test_tok = test_ds.map(preprocess, batched=True, remove_columns=remove_cols)
train_tok

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1600
})

## Model + training

In [5]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

cer_metric = evaluate.load("cer")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    exact_match = np.mean(
        [p == l for p, l in zip(decoded_preds, decoded_labels)]
    )
    cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"exact_match": exact_match, "cer": cer}

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [6]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [7]:
trainer.train()

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Exact Match,Cer
1,0.698886,0.615139,0.485000,0.179398
2,0.024110,0.000238,1.000000,0.000000
3,0.018707,0.000137,1.000000,0.000000
4,0.002862,0.000318,1.000000,0.000000
5,0.005090,0.000041,1.000000,0.000000
6,0.000964,0.000028,1.000000,0.000000
7,0.003729,0.000025,1.000000,0.000000
8,0.002013,0.000021,1.000000,0.000000
9,0.001198,0.000019,1.000000,0.000000
10,0.000758,0.000019,1.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1000, training_loss=0.1357996719190851, metrics={'train_runtime': 285.3643, 'train_samples_per_second': 56.069, 'train_steps_per_second': 3.504, 'total_flos': 146760553463808.0, 'train_loss': 0.1357996719190851, 'epoch': 10.0})

## Evaluate on held-out test split

In [8]:
test_metrics = trainer.evaluate(eval_dataset=test_tok, metric_key_prefix="test")
test_metrics

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Exact Match,Cer
0.000758,0.000298,10,1.000000,0.000000


{'test_loss': 0.0002978252014145255, 'test_exact_match': 1.0, 'test_cer': 0.0}

In [9]:
trainer.save_model(str(OUTPUT_DIR / "best"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "best"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/disfluency_repair_indot5_small/best/tokenizer_config.json',
 '../models/disfluency_repair_indot5_small/best/tokenizer.json')

## Sanity check inference

In [10]:
model.eval()
model.to(device)


def repair(text: str) -> str:
    inputs = tokenizer(TASK_PREFIX + text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)


samples = [r["input"] for r in test_rows[:10]]
targets = [r["target"] for r in test_rows[:10]]

for src, tgt in zip(samples, targets):
    pred = repair(src)
    print(f"raw:    {src}")
    print(f"target: {tgt}")
    print(f"pred:   {pred}")
    print("-" * 60)

raw:    kak ayam gorengnya jadi satu porsi aja deh, kebanyakan tadi
target: kak ayam gorengnya jadi satu porsi aja deh kebanyakan tadi
pred:   kak ayam gorengnya jadi satu porsi aja deh kebanyakan tadi
------------------------------------------------------------
raw:    gak setuju sama itu, ada yang kelewat dicatet kak
target: gak setuju sama itu ada yang kelewat dicatet kak
pred:   gak setuju sama itu ada yang kelewat dicatet kak
------------------------------------------------------------
raw:    bu tambah es teh manis satu lagi ya, lupa tadi minta
target: bu tambah es teh manis satu lagi ya lupa tadi minta
pred:   bu tambah es teh manis satu lagi ya lupa tadi minta
------------------------------------------------------------
raw:    eh mbak, gue lagi liatin menu, ulang lagi dong yang barusan
target: eh mbak gue lagi liatin menu ulang lagi dong yang barusan
pred:   eh mbak gue lagi liatin menu ulang lagi dong yang barusan
------------------------------------------------------------
r